In [2]:
# Model takes a list of sentences and outputs an array of score with a formality score fore each sentence.




In [21]:
# Imports
import numpy as np

# The word2vec imports

import gensim
import gensim.downloader as api
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# The LSTM imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import TransformerEncoder, TransformerEncoderLayer

In [34]:
class BidirectionalTransformer(nn.Module):
    def __init__(self, input_dim, model_dim, num_heads, num_layers, dropout=0.1):
        super(BidirectionalTransformer, self).__init__()
        
        # Positional encoding (optional if you're working with sequences)
        self.positional_encoding = nn.Embedding(5000, model_dim)  # You can customize the maximum length
        
        # Transformer encoder layers
        encoder_layer = TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dropout=dropout)
        self.transformer_encoder = TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Linear layer for output (classification, regression, etc.)
        self.fc = nn.Linear(model_dim, input_dim)  # Adjust output size as needed
        
    def forward(self, x):
        # Optionally add positional encoding (for sequences)
        seq_len, batch_size = x.size(0), x.size(1)
        positions = torch.arange(0, seq_len).unsqueeze(1).expand(seq_len, batch_size).to(x.device)
        x = x + self.positional_encoding(positions)
        
        # Transformer Encoder (bidirectional)
        encoded = self.transformer_encoder(x)
        
        # Output through linear layer
        output = self.fc(encoded)
        
        return output
    

    

def sentences_to_vectors(sentences, model):
    encoding_dim = model.vector_size
    no_sentence = len(sentences)
    max_word_count = 25
    
    # Initialize a 3D array with zeros
    returned_array = np.zeros((encoding_dim, max_word_count, no_sentence))

    def tokenize(sentence):
        tokens = word_tokenize(sentence)  # Tokenize sentence
        return [word for word in tokens if word not in stopwords.words('english')]  # Remove stopwords

    for i, sentence in enumerate(sentences):
        words = tokenize(sentence)
        word_vectors = [model[word] for word in words if word in model]
        
        # Pad or truncate word_vectors to fit max_word_count
        if len(word_vectors) < max_word_count:
            # Pad with zeros if there are fewer than max_word_count word vectors
            padded_vectors = np.array(word_vectors + [[0] * encoding_dim] * (max_word_count - len(word_vectors)))
        else:
            # Truncate if there are more than max_word_count word vectors
            padded_vectors = np.array(word_vectors[:max_word_count])
        
        # Fill the 3D array
        returned_array[:, :, i] = padded_vectors.T  # Transpose to match shape (encoding_dim, max_word_count)

    return returned_array

In [8]:
print(list(api.info()['models'].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [10]:
model = api.load('glove-twitter-25')

[==================================================] 100.0% 104.8/104.8MB downloaded


In [16]:
example_preprocessed_sentences = [
    "team wanted provide update latest progress marketing campaign",
    "successfully completed initial phase client happy results far",
    "however changes need addressed move forward",
    "arrange meeting next week go revised strategy ensure page",
    "please let know availability",
    "looking forward continued collaboration"
]


In [35]:
sentence_vectors = sentences_to_vectors(example_preprocessed_sentences, model)
print(sentence_vectors.shape)

(25, 25, 6)
